# Example 9: Sherman et al., 2018 - Dual Domain Mass Transfer (DDMT) in 1-D Column Experiments with adsorption of Na in mobile and immobile domains.

This example notebook models physical experiments published in Sherman et al., 2018 https://agupubs.onlinelibrary.wiley.com/doi/full/10.1029/2018WR023420

 The experiments include a NaCl tracer through a column packed with a Na-dominated zeolite clinoptilolite porous medium that contained intragranular pore space within the each grain (immobile domain) in addition to intergranular porosity between each grain (mobile domain). Data were collected at three downstream sampling ports, but this example will focus on the first sampling port at about 0.21 m. The observation data were collected in the form of fluid electrical conductivity because it is cost-efficient and relatively instantaneous. 

 It is assumed that the fluid electrical conductivity is preferentially sampling the mobile domain of the dual-domain system, so history matching will focus on fitting the data with the simulated mobile domain with the numerical simulators.

 For the reactive transport modeling, the fluid electrical conducitivity signature will need to be decomposed into to two signatures for simulation; Na & Cl. Cl is assumed to transport in a non-reactive fashion, but is allowed to diffuse into and out of immobile pore space. Na is assumed to transport in a reactive fashion, with the ability to sorb in both mobile and mobile domains, in addition being able to diffuse into and out of immobile pore space. After the reactive transport simulation of these two constituents, their signatures are recomposed into their composite electrical signature for the mobile domain and compared to the observed fluid electrical conducitivity data collected during the experiments. 

 This notebook will walk through simulation of the experiment with mf6rtm to compare to PHREEQC simulation results as a benchmark for DDMT. The input parameters are based on base realization parameters calibrated by means of pestpp-ies. The calibration process was used as a polishing step for the parameterization as this medium with this tracer has been well characterized in the literature by Briggs et al., 2014 https://agupubs.onlinelibrary.wiley.com/doi/full/10.1002/2014WR015880 and Swanson et al., 2012 https://pubs.usgs.gov/publication/70074266. The PHREEQC set-up and calibration process with pestpp-ies will not be included here for brevity. Measured data and PHREEQC simulation files are included in the PHREEQC folder within the ex9 folder for reference.

In [ ]:
# import libraries, set-up folder structures, etc.
from pathlib import Path
import os
import sys
sys.path.insert(0, os.path.abspath(os.path.join('..', 'src')))
from datetime import datetime
import shutil

import flopy
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from mf6rtm import utils, mup3d
import mf6rtm
import inspect

prefix = "ex9"
dataws = Path.cwd() / "data"
databasews = Path.cwd() / "database"
mf6_ws = Path.cwd() / prefix / "mf6rtm"
phreeqc_ws = Path.cwd() / prefix / "phreeqc"
binws = Path.cwd() / "bin"

# Physical Parameter Definitions

In [ ]:
# define physical experiment parameters for simulator
seconds_per_day = 86400.0

length_units = "meters"
time_units = "days"

# Higher refinement is affordable because this branch only simulates the first 0.25 m.
refinement_factor = 5

nlay = 1
nrow = 1

# column size parameters
physical_Lx = 0.25 # meters
column_diameter_m = 5.2 / 100.0 # meters
area = np.pi * (column_diameter_m / 2.0) ** 2 # meters

# average linear pore velocity for consistency with PHREEQC.
linear_velocity_cm_s = 0.0077 # centimeters per second
linear_velocity_m_per_d = linear_velocity_cm_s * 0.01 * seconds_per_day # meters per day

# PHREEQC reference pulse timing.
shift_seconds = 158.0
shift_days = shift_seconds / seconds_per_day
pulse_shifts = 5
flush_shifts = 100
pulse_seconds = pulse_shifts * shift_seconds
flush_seconds = flush_shifts * shift_seconds

# Original PHREEQC-equivalent cell length from velocity * shift time.
phreeqc_delr_from_velocity = linear_velocity_cm_s * shift_seconds / 100.0

# Refined grid over exactly 0.25 m.
target_delr = phreeqc_delr_from_velocity / refinement_factor
ncol = int(np.ceil(physical_Lx / target_delr))
nxyz = nlay * nrow * ncol

delr = physical_Lx / ncol
Lx = physical_Lx

# Choose timestep counts to keep Courant close to 1 while preserving pulse/flush durations.
target_dt_seconds = delr / (linear_velocity_m_per_d / seconds_per_day)
pulse_nstp = max(1, int(round(pulse_seconds / target_dt_seconds)))
flush_nstp = max(1, int(round(flush_seconds / target_dt_seconds)))

nper = 2
perlen_pulse = pulse_seconds / seconds_per_day
perlen_flush = flush_seconds / seconds_per_day
tdis_rc = [
    (perlen_pulse, pulse_nstp, 1.0),
    (perlen_flush, flush_nstp, 1.0),
]

dt_seconds_pulse = pulse_seconds / pulse_nstp
dt_seconds_flush = flush_seconds / flush_nstp
dt_days = dt_seconds_pulse / seconds_per_day

delc = 1.0
top = 0.0
botm = -area

# PHREEQC -stagnant 1 5e-05 0.46 0.2, with calibrated notebook values.
prsity = 0.5
theta_mobile = np.full(nxyz, 0.5, dtype=float)
theta_immobile = np.full(nxyz, 0.209, dtype=float)
alpha_stagnant_per_s = 5.9e-5
alpham = np.full(nxyz, alpha_stagnant_per_s * seconds_per_day, dtype=float)

# physical flow rate implied by the target velocity, porosity, and column area.
q = prsity * linear_velocity_m_per_d * area
flow_ml_min = q / seconds_per_day * 1.0e6 * 60.0
courant_pulse = linear_velocity_m_per_d * (dt_seconds_pulse / seconds_per_day) / delr
courant_flush = linear_velocity_m_per_d * (dt_seconds_flush / seconds_per_day) / delr

# Flow setup.
k11 = 100.0
k33 = k11
icelltype = 1
strt = np.ones((nlay, nrow, ncol), dtype=float)

ibound = np.ones((nlay, nrow, ncol), dtype=int)
ibound[0, 0, -1] = -1

# ghb inlet intended to approximate target flow q through the 0.25 m domain.
outlet_head = 1.0
ghb_cell = (0, 0, 0)
chd_cell = (0, 0, ncol - 1)
ghb_cond = k11 * area / (0.5 * delr)
ghb_bhead = outlet_head + q * (Lx - 0.5 * delr) / (k11 * area)

chdspd = {
    0: [[chd_cell, outlet_head]],
    1: [[chd_cell, outlet_head]],
}
ghb_spd = {
    0: [[ghb_cell, ghb_bhead, ghb_cond]],
    1: [[ghb_cell, ghb_bhead, ghb_cond]],
}

# PHREEQC dispersivity and diffusion coefficient, with calibrated notebook values.
#dispersivity = 0.003
dispersivity = 0.0015
molecular_diffusion = 3.8e-9 * seconds_per_day

# Solver controls.
nouter, ninner = 100, 300
hclose, rclose, relax = 1e-8, 1e-8, 1.0

# Sample port / PHREEQC comparison mapping.
sample_distance_m = 0.21
sample_phreeqc_cell = 17
sample_cell = int(np.clip(round(sample_distance_m / delr), 1, ncol))
sample_cell_distance_m = sample_cell * delr
sample_cell_label = f"0.21 m port / MF6RTM cell {sample_cell}"

# Map original PHREEQC punch-cell distances where they fall inside the short domain.
phreeqc_punch_cells = [1, 17, 18]
model_punch_cells = [
    int(np.clip(round((c * phreeqc_delr_from_velocity) / delr), 1, ncol))
    for c in phreeqc_punch_cells
]

print(f"folder prefix: {prefix}")
print(f"Refinement factor: x{refinement_factor}")
print(f"Short physical length: {Lx:.4f} m")
print(f"Column diameter: {column_diameter_m:.4f} m; area: {area:.6e} m2")
print(f"MF6 cells: {ncol} (delr={delr:.8f} m)")
print(f"Target velocity: {linear_velocity_cm_s:.4f} cm/s = {linear_velocity_m_per_d:.4f} m/d")
print(f"Flow q: {q:.6e} m3/d = {flow_ml_min:.3f} mL/min")
print(f"Pulse duration: {perlen_pulse:.8f} d with {pulse_nstp} steps; dt={dt_seconds_pulse:.3f} s")
print(f"Flush duration: {perlen_flush:.8f} d with {flush_nstp} steps; dt={dt_seconds_flush:.3f} s")
print(f"GHB bhead: {ghb_bhead:.6f}; GHB conductance: {ghb_cond:.6g}")
print(f"Courant pulse/flush: {courant_pulse:.3f}, {courant_flush:.3f}")
print(f"DDMT zetaim: {alpham[0]:.4f} 1/d")
print(f"Sample mapping: {sample_cell_label} at {sample_cell_distance_m:.4f} m")
print(f"Mapped PHREEQC cells inside short domain: {list(zip(phreeqc_punch_cells, model_punch_cells))}")

# Chemistry Initilization and Postfix Definitions

In [ ]:
# PHREEQC solutions converted from mmol/L to approximate mol/kgw for Mup3d.
solutions = {
    "pH": [5.5, 5.5, 5.5],
    "pe": [13.0, 13.0, 13.0],
    "Na": [1.19e-3, 8.0e-3, 1.19e-3],
    "Cl": [1.19e-3, 8.0e-3, 1.19e-3],
}

sol_ic = 1
solution = mup3d.Solutions(solutions)
solution.set_ic(sol_ic)

if mf6_ws.exists():
    shutil.rmtree(mf6_ws)

model = mup3d.Mup3d(prefix, solution, nlay, nrow, ncol)
model.set_wd(mf6_ws)
model.set_database(databasews / "pht3d_datab.dat")

# KINETICS 1 for all cells. The reactant is named Na so the current Mup3d
# database-name check accepts it; the rate law below is the original Na_sorption law.
kinetics = mup3d.KineticPhases(
    {
        0: {
            "Na": {
                "formula": "Na+ 1.0",
                "m": 0.0195,
                "tol": 1.0e-12,
            }
        }
    }
)
kinetics.set_ic(1)
model.set_phases(kinetics)

km = 0.0001327 # 1/second
kd = 0.01 # L/kg
solids = 2000.0 # kg water per kg solid
# Definition of Na kintetically controlled sorption
Path(prefix).mkdir(parents=True, exist_ok=True)
postfix = Path(prefix) / "na_adsorption_selected_output.phqr"
postfix.write_text(
    f"""KNOBS
    -iterations 1000

RATES
Na
-start
10 km = {km} # 1/s
20 kd = {kd} # L/kg
30 solids = {solids} # kg solids
40 C = MOL(\"Na+\")
50 rate = -km * (C - (M / solids) / kd)
60 moles = rate * TIME
70 if (M - moles) < 0 then moles = M
80 SAVE moles
-end

SELECTED_OUTPUT 1
    -high_precision true
    -reset false
    -solution true
    -totals Na Cl
    -molalities Na+ Cl-
    -step true
    -ionic_strength true
    -time true
    -user_punch true

USER_PUNCH 1
    -headings time_d cell pH pe Na Cl Na_molal Cl_molal KIN_Na EC_uScm_calc
10 PUNCH SIM_TIME / 86400
20 PUNCH CELL_NO
30 PUNCH -LA(\"H+\")
40 PUNCH -LA(\"e-\")
50 PUNCH TOT(\"Na\")
60 PUNCH TOT(\"Cl\")
70 PUNCH MOL(\"Na+\")
80 PUNCH MOL(\"Cl-\")
90 PUNCH KIN(\"Na\")
100 PUNCH 1000 * (50.1 * TOT(\"Na\") + 76.3 * TOT(\"Cl\")) # conversion to fluid electrical conductivity in uS per cm
END
""",
    encoding="utf-8",
)
model.set_postfix(postfix)

# Initialize Model

In [ ]:
# Initializes PhreeqcRM, discovers components, writes mf6rtm.yaml, and creates model.sconc.
model.initialize()

# Keep the notebook compatible with branches that use either nxyz or ncpl
# for the total grid-cell count.
if not hasattr(model, "nxyz"):
    model.nxyz = getattr(model, "ncpl", nxyz)
if not hasattr(model, "ncpl"):
    model.ncpl = getattr(model, "nxyz", nxyz)

print("Components:", model.components)

# Upstream GHB Chemistry Definition

In [ ]:
# stress period 0: high-NaCl pulse solution.
# stress period 1: low-NaCl flush solution.
ghbchem = mup3d.ChemStress("ghb")
ghbchem.set_spd({
    0: [2],
    1: [3],
})
model.set_chem_stress(ghbchem)

# Dual-Domain Mass Transfer (DDMT) Initialization

In [ ]:
model.set_ddmt(
    theta_mobile=theta_mobile,
    theta_immobile=theta_immobile,
    alpham=alpham,
    mode="reactive",
    immobile_initial="mobile",
    immobile_yaml=model.phreeqcyaml_file,
    output=True,
    output_format="csv",
)

model.config.reactive["enabled"] = True
model.config.output["output_format"] = "csv"

model.save_config()
print((Path(model.wd) / "mf6rtm.toml").read_text())

# Build MF6 Model Object

In [ ]:
# helper fxn to build ghb package with respective flux and chemistry
def build_ghb_spd_with_chem(model):
    """Combine base GHB records with Mup3d ChemStress auxiliary chemistry."""
    if isinstance(model.ghb.data, dict):
        ghb_spd_with_chem = {}
        for sp, base_records in ghb_spd.items():
            chem_records = model.ghb.data[sp]
            records = []
            for i, rec in enumerate(base_records):
                row = [rec[0], rec[1], rec[2]]
                row.extend(chem_records[i])
                records.append(row)
            ghb_spd_with_chem[sp] = records
        return ghb_spd_with_chem

    records = []
    for i, rec in enumerate(ghb_spd[0]):
        row = [rec[0], rec[1], rec[2]]
        row.extend(model.ghb.data[i])
        records.append(row)
    return records

# fxn to build mf6 model
def build_model(model):


    gwfname = "gwf"
    sim_ws = model.wd
    exe_path = os.path.join(sim_ws, "mf6")
    sim = flopy.mf6.MFSimulation(sim_name=model.name, sim_ws=sim_ws, exe_name=exe_path, version="mf6")

    flopy.mf6.ModflowTdis(sim, nper=nper, perioddata=tdis_rc, time_units=time_units)

    gwf = flopy.mf6.ModflowGwf(
        sim,
        modelname=gwfname,
        save_flows=True,
        model_nam_file=f"{gwfname}.nam",
    )

    imsgwf = flopy.mf6.ModflowIms(
        sim,
        print_option="SUMMARY",
        outer_dvclose=hclose,
        outer_maximum=nouter,
        under_relaxation="NONE",
        inner_maximum=ninner,
        inner_dvclose=hclose,
        rcloserecord=rclose,
        linear_acceleration="CG",
        scaling_method="NONE",
        reordering_method="NONE",
        relaxation_factor=relax,
        filename=f"{gwfname}.ims",
    )
    sim.register_ims_package(imsgwf, [gwf.name])

    dis = flopy.mf6.ModflowGwfdis(
        gwf,
        length_units=length_units,
        nlay=nlay,
        nrow=nrow,
        ncol=ncol,
        delr=delr,
        delc=delc,
        top=top,
        botm=botm,
        idomain=np.ones((nlay, nrow, ncol), dtype=int),
        filename=f"{gwfname}.dis",
    )
    dis.set_all_data_external()

    npf = flopy.mf6.ModflowGwfnpf(
        gwf,
        save_flows=True,
        save_saturation=True,
        icelltype=icelltype,
        k=k11,
        k33=k33,
        save_specific_discharge=True,
        filename=f"{gwfname}.npf",
    )
    npf.set_all_data_external()

    flopy.mf6.ModflowGwfic(gwf, strt=strt, filename=f"{gwfname}.ic")

    chd = flopy.mf6.ModflowGwfchd(
        gwf,
        maxbound=1,
        stress_period_data=chdspd,
        save_flows=False,
        pname="CHD",
        filename=f"{gwfname}.chd",
    )
    chd.set_all_data_external()

    ghb = flopy.mf6.ModflowGwfghb(
        gwf,
        stress_period_data=build_ghb_spd_with_chem(model),
        save_flows=True,
        auxiliary=model.components,
        pname="ghb",
        filename=f"{gwfname}.ghb",
    )
    ghb.set_all_data_external()

    flopy.mf6.ModflowGwfoc(
        gwf,
        head_filerecord=f"{gwfname}.hds",
        budget_filerecord=f"{gwfname}.cbb",
        headprintrecord=[("COLUMNS", 10, "WIDTH", 15, "DIGITS", 6, "GENERAL")],
        saverecord=[("HEAD", "ALL"), ("BUDGET", "ALL")],
        printrecord=[("HEAD", "LAST"), ("BUDGET", "LAST")],
    )

    for component in model.components:
        print(f"Setting model for component: {component}")
        gwtname = component

        gwt = flopy.mf6.MFModel(
            sim,
            model_type="gwt6",
            modelname=gwtname,
            model_nam_file=f"{gwtname}.nam",
        )

        imsgwt = flopy.mf6.ModflowIms(
            sim,
            print_option="SUMMARY",
            outer_dvclose=hclose,
            outer_maximum=nouter,
            under_relaxation="NONE",
            inner_maximum=ninner,
            inner_dvclose=hclose,
            rcloserecord=rclose,
            linear_acceleration="BICGSTAB",
            scaling_method="NONE",
            reordering_method="NONE",
            relaxation_factor=relax,
            filename=f"{gwtname}.ims",
        )
        sim.register_ims_package(imsgwt, [gwt.name])

        dis = flopy.mf6.ModflowGwtdis(
            gwt,
            length_units=length_units,
            nlay=nlay,
            nrow=nrow,
            ncol=ncol,
            delr=delr,
            delc=delc,
            top=top,
            botm=botm,
            idomain=np.ones((nlay, nrow, ncol), dtype=int),
            filename=f"{gwtname}.dis",
        )
        dis.set_all_data_external()

        ic = flopy.mf6.ModflowGwtic(gwt, strt=model.sconc[component], filename=f"{gwtname}.ic")
        ic.set_all_data_external()

        ssm = flopy.mf6.ModflowGwtssm(
            gwt,
            sources=["ghb", "aux", component],
            filename=f"{gwtname}.ssm",
        )
        ssm.set_all_data_external()

        flopy.mf6.ModflowGwtadv(gwt, scheme="tvd")

        alpha_l = np.ones((nlay, nrow, ncol), dtype=float) * dispersivity
        alpha_th = np.zeros((nlay, nrow, ncol), dtype=float)
        alpha_tv = np.zeros((nlay, nrow, ncol), dtype=float)

        dsp = flopy.mf6.ModflowGwtdsp(
            gwt,
            xt3d_off=True,
            alh=alpha_l,
            ath1=alpha_th,
            atv=alpha_tv,
            diffc=molecular_diffusion,
            filename=f"{gwtname}.dsp",
        )
        dsp.set_all_data_external()

        mst = flopy.mf6.ModflowGwtmst(
            gwt,
            porosity=prsity,
            first_order_decay=None,
            filename=f"{gwtname}.mst",
        )
        mst.set_all_data_external()

        flopy.mf6.ModflowGwtoc(
            gwt,
            budget_filerecord=f"{gwtname}.cbb",
            concentration_filerecord=f"{gwtname}.ucn",
            concentrationprintrecord=[("COLUMNS", 10, "WIDTH", 15, "DIGITS", 10, "GENERAL")],
            saverecord=[("CONCENTRATION", "ALL")],
            printrecord=[("CONCENTRATION", "LAST")],
        )

        flopy.mf6.ModflowGwfgwt(
            sim,
            exgtype="GWF6-GWT6",
            exgmnamea=gwfname,
            exgmnameb=gwtname,
            filename=f"{gwtname}.gwfgwt",
        )

    sim.write_simulation()

    # prep bins
    utils.prep_bins(
        sim_ws,
        src_path=os.path.join("bin"),
        get_only=["mf6", "libmf6"],
    )
    return sim

# Build mf6 model

In [ ]:
sim = build_model(model)

# Run mf6 Model

In [ ]:
model.run()

# Plot Breakthrough of Na & Cl in both Mobile and Immobile Domains

In [ ]:
ddmt_path = Path(model.wd) / "ddmt.csv"
ddmt = pd.read_csv(ddmt_path)

print(ddmt.columns.tolist())
display(ddmt.head())

vars_to_plot = ["Na", "Cl"]

mobile = ddmt.loc[(ddmt["domain"] == "mobile") & (ddmt["cell"] == sample_cell)].sort_values("time_d")
immobile = ddmt.loc[(ddmt["domain"] == "immobile") & (ddmt["cell"] == sample_cell)].sort_values("time_d")

# plot figure
fig, ax = plt.subplots(figsize=(8, 4))

for var in vars_to_plot:
    if var in mobile.columns:
        ax.plot(mobile["time_d"] * seconds_per_day, mobile[var], marker="o", ms=2.5, lw=1.3, label=f"{var} mobile")

    if var in immobile.columns:
        ax.plot(immobile["time_d"] * seconds_per_day, immobile[var], linestyle="--", marker="s", ms=2.5, lw=1.3, label=f"{var} immobile")

ax.set_xlabel("time (seconds)")
ax.set_ylabel("concentration (mol/L)")
ax.set_title(f"DDMT breakthrough at {sample_cell_label} ({sample_cell_distance_m:.3f} m)")
ax.legend()
fig.tight_layout()


# Plot Breakthrough of Fluid Electrical Conductivity in Mobile and Immobile Domains

In [ ]:
def add_ec_uscm_from_na_cl(df, na_col="Na", cl_col="Cl", ec_col="EC_uScm_calc"):
    """Add PHREEQC-style calculated EC from Na and Cl concentrations in mol/L."""
    out = df.copy()

    missing = [col for col in (na_col, cl_col) if col not in out.columns]
    if missing:
        raise KeyError(f"Missing required concentration column(s): {missing}")

    out[ec_col] = 1000.0 * (50.1 * out[na_col] + 76.3 * out[cl_col])
    return out


def plot_ec_breakthrough(ddmt, cells=None, domains=("mobile", "immobile"), time_scale=seconds_per_day, time_label="time (seconds)"):
    """Plot calculated electrical conductivity from ddmt.csv for selected cells/domains."""
    if cells is None:
        cells = (sample_cell,)

    ddmt_ec = add_ec_uscm_from_na_cl(ddmt)

    # plot figure
    fig, ax = plt.subplots(figsize=(8, 4))

    for cell in cells:
        for domain in domains:
            subset = ddmt_ec.loc[(ddmt_ec["cell"] == cell) & (ddmt_ec["domain"] == domain)].sort_values("time_d")

            if subset.empty:
                continue

            style = "-" if domain == "mobile" else "--"
            ax.plot(subset["time_d"] * time_scale, subset["EC_uScm_calc"], linestyle=style, lw=1.5, label=f"cell {cell} {domain}")

    ax.set_xlabel(time_label)
    ax.set_ylabel("calculated EC (uS/cm)")
    ax.set_title("Calculated electrical conductivity from Na and Cl")
    ax.legend()
    fig.tight_layout()

    return ddmt_ec, fig, ax


ddmt_ec, fig, ax = plot_ec_breakthrough(ddmt, cells=(sample_cell,), domains=("mobile", "immobile"))

display(ddmt_ec.head())

# Compare PHREEQC & MF6RTM Simulated Values to Observed Fluid Electrical Conductivity

In [ ]:
def load_pest_phreeqc_ec_results(phreeqc_dir=None):
    """Return measured and PHREEQC simulated EC data from obs.csv and sim.csv."""
    if phreeqc_dir is None:
        phreeqc_dir = Path(prefix) / "phreeqc"
    phreeqc_dir = Path(phreeqc_dir)

    obs = pd.read_csv(phreeqc_dir / "obs.csv")
    sim = pd.read_csv(phreeqc_dir / "sim.csv")

    obs_ec = (obs[["time_mins", "conc"]].rename(columns={"time_mins": "time_minutes", "conc": "measured_ec_uscm"}).copy())
    sim_ec = (sim[["time_mins", "conc"]].rename(columns={"time_mins": "time_minutes", "conc": "modeled_ec_uscm"}).copy())

    pest_ec = (pd.merge(obs_ec, sim_ec, on="time_minutes", how="outer").sort_values("time_minutes").reset_index(drop=True))

    return pest_ec


def mf6rtm_ec_series_from_ddmt(ddmt, cell, domain="mobile", time_col="time_minutes"):
    """Return MF6RTM EC time series for one DDMT cell/domain."""
    ddmt_ec = add_ec_uscm_from_na_cl(ddmt)
    series = ddmt_ec.loc[(ddmt_ec["cell"] == cell) & (ddmt_ec["domain"] == domain)].copy()

    if series.empty:
        raise ValueError(f"No DDMT rows found for cell={cell}, domain={domain!r}")

    series[time_col] = series["time_d"] * 24.0 * 60.0
    return series.sort_values(time_col)


def compare_mf6rtm_to_pest_phreeqc_ec(ddmt, cell=sample_cell, domain="mobile", include_immobile=True, cell_label=None, phreeqc_dir=None):
    """Plot measured EC, PHREEQC modeled EC, and MF6RTM EC."""
    pest_ec = load_pest_phreeqc_ec_results(phreeqc_dir=phreeqc_dir)
    mf6_ec = mf6rtm_ec_series_from_ddmt(ddmt, cell=cell, domain=domain)

    compare = pest_ec.copy()
    compare["mf6rtm_ec_uscm"] = np.interp(compare["time_minutes"], mf6_ec["time_minutes"], mf6_ec["EC_uScm_calc"], left=np.nan, right=np.nan)
    compare["mf6rtm_minus_measured"] = (compare["mf6rtm_ec_uscm"] - compare["measured_ec_uscm"])
    compare["phreeqc_minus_measured"] = (compare["modeled_ec_uscm"] - compare["measured_ec_uscm"])

    # calc stats
    metric_rows = []
    for model_col, residual_col in [("modeled_ec_uscm", "phreeqc_minus_measured"), ("mf6rtm_ec_uscm", "mf6rtm_minus_measured")]:
        valid = compare[[model_col, residual_col]].dropna()
        metric_rows.append(
            {
                "series": model_col,
                "n": len(valid),
                "bias_uscm": valid[residual_col].mean(),
                "mae_uscm": valid[residual_col].abs().mean(),
                "rmse_uscm": np.sqrt(np.mean(valid[residual_col] ** 2)),
            }
        )
    metrics = pd.DataFrame(metric_rows)

    # plot figure
    fig, ax = plt.subplots(figsize=(9, 4.8))

    ax.scatter(pest_ec["time_minutes"], pest_ec["measured_ec_uscm"], s=18, facecolor="None", edgecolor="red", label="measured EC", zorder=4)
    ax.plot(pest_ec["time_minutes"], pest_ec["modeled_ec_uscm"], color="dodgerblue", lw=3.0, label="PHREEQC modeled EC w/pestpp-ies")
    ax.plot(mf6_ec["time_minutes"], mf6_ec["EC_uScm_calc"], color="black", linestyle='--', lw=1.5, label=f"MF6RTM modeled EC")

    if include_immobile and domain != "immobile":
        try:
            immobile_ec = mf6rtm_ec_series_from_ddmt(ddmt, cell=cell, domain="immobile")
            ax.plot(immobile_ec["time_minutes"], immobile_ec["EC_uScm_calc"], color="tab:blue", lw=1.4, ls="--", alpha=0.75, label=f"MF6RTM immobile EC, cell {cell}")
        except ValueError:
            pass

    ax.set_xlabel("time (minutes)")
    ax.set_ylabel("electrical conductivity (uS/cm)")
    if cell_label is None:
        cell_label = f"cell {cell}"
    ax.set_title(f"EC comparison at {cell_label}")
    ax.set_xlim([0, 450])
    ax.legend()
    fig.tight_layout()

    return compare, metrics, fig, ax


ec_compare, ec_metrics, fig, ax = compare_mf6rtm_to_pest_phreeqc_ec(ddmt, cell=sample_cell, domain="mobile", include_immobile=False, cell_label=sample_cell_label)

display(ec_metrics)
display(ec_compare.head())
